# I2C device initialization

In [1]:
%matplotlib tk
from equipment.equipment_init import *
from project.sc8563 import project
import datetime
from time import sleep

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8563", revision="1p1", emulator="ch341", logging=False, is_gui=False)

initialized the dm6500 connection
failed to initialize the dm6500
failed to initialize n6705
failed to initialize tektronix dp series
failed to initialize asd-906b
failed to initialize it8511a
initialized with i2c address 0x6e by default


# Preset

In [107]:
def preset():
    ic.FSW_SET = 0
    ic.SS_TIMEOUT = 0
    ic.SET_IBAT_SNS_RES = 0
    ic.PMID_IN_RANGE_DIS = 1
    ic.STANDBY_MODE_SET = 1
    ic.EXT1_OVP = 7 # max 19V
    ic.EXT2_OVP = 7 # max 19V
    ic.IIN_OCP = 170 # Max 6.375A
    ic.IIN_REG = 160 # Max 6A
    ic.IIN_UCP_DIS = 1
    ic.VBAT_OVP = 0xF0 # Max 4.6V
    ic.VIN_OVP = 7 # 16.5V
    ic.VOUT_OVP = 3 # 5.3V
    ic.C1P2OUT_OVP = 7
    ic.C1P2OUT_UVP = 7
    ic.VBAT_OVP_DIS = 1
    ic.IIN_REG_DIS = 1
    ic.IIN_OCP_DIS = 1
    ic.IIN_UCP_DIS = 1
    ic.VBAT_REG_DIS = 1
    ic.IBAT_REG_DIS = 1
    ic.IBAT_OCP_DIS = 1
    ic.NTC_FLT_DIS = 1
    ic.TDIE_REG_DIS = 1
    ic.ADC_EN = 1

def decrease_iout(target_vout, iout_max, mode_n):

    list_rev_iout = list(i for i in range(1, iout_max+1))
    list_rev_iout.reverse()

    # meas_vin  = ps.ch1.voltage
    meas_vin  = dm1.voltage_100
    offset = abs(meas_vin - target_vout*mode_n) / iout_max

    for set_iout in list_rev_iout:
        
        estimate_rev_vin = target_vout* mode_n + offset*set_iout
        ps.ch1.vset = estimate_rev_vin
        ld.cc.iset = set_iout
        print(f"VIN={estimate_rev_vin:.03f}, ILOAD={set_iout:.01f}")
        sleep(0.5)


In [30]:
dm1.voltage_100 # VIN

12.06742

In [31]:
dm2.voltage_10 # VOUT

3.999168

In [4]:
ic.ADC_EN = 1
ic.ADC_RATE = 0

ic.status_adc

ADC        Hex      Value
---------  -----  -------
IIN_ADC    0xfff  7.67812
VIN_ADC    0xfff   20.475
VEXT1_ADC  0xfff   20.475
VEXT2_ADC  0xfff   20.475
VOUT_ADC   0xfff  5.11875
VBAT_ADC   0xfff  5.11875
IBAT_ADC   0xfff  12.7969
C1P_ADC    0xfff   20.475
NTC_ADC    0xfff  59.9917
TDIE_ADC   0x3ff    511.5


# FSW_SET[3:0] sweep
# IOUT : up to 12A / 0.5A steps
# VOUT target : 4.4V
# IIN_REG_DIS = 1
# VBAT_REG_DIS = 1


# efficiency measurement

In [116]:
mode_n = 2

preset()

if mode_n == 3:
    ic.MODE = 0 # 3TO1 FWD
elif mode_n ==2:
    ic.MODE = 1 # 2TO1 FWD
elif mode_n ==1:
    ic.MODE = 2 # 1TO1 FWD
else:
    raise("error")

# init_vout = 3.3

# target_vout = 3.6
# target_vout = 4.0
target_vout = 4.4

init_vout = target_vout - 0.2

bs.vset = init_vout

vin_init = init_vout * mode_n + 0.2
ps.ch1.cfg_all = vin_init, 5.3

ic.CP_EN = 1

In [124]:
ic.CP_EN=0

In [89]:
ld.disable

In [95]:
ld.voltage
ld.disable
ld.cc.iset = 1

In [118]:

ic.status

Addr    Reg           Value    Bit7                  Bit6               Bit5               Bit4                Bit3                  Bit2                  Bit1                  Bit0
------  ------------  -------  --------------------  -----------------  -----------------  ------------------  --------------------  --------------------  --------------------  -------------------
0x01    INT_DEVICE0   0x00     POR_FLAG              CP_SWITCHING_FLAG  QB_ON_FLAG         VIN_PRESENT_FLAG    VOUT_INSERT_FLAG      VOUT_TH_REV_EN_FLAG   VOUT_TH_CHG_EN_FLAG   VIN_TH_CHG_EN_FLAG
0x02    INT_DEVICE1   0x00     VEXT1_REMOVE_FLAG     VEXT2_REMOVE_FLAG  VEXT1_INSERT_FLAG  VEXT2_INSERT_FLAG   VEXT1_OVP_FLAG        VEXT2_OVP_FLAG        VEXT1_DRV_ON_FLAG     VEXT2_DRV_ON_FLAG
0x03    INT_DEVICE2   0x00     ADC_DONE_FLAG         IIN_UCP_RISE_FLAG  RESERVED           TDIE_REG_EXIT_FLAG  TDIE_REG_ACTIVE_FLAG  VBAT_REG_ACTIVE_FLAG  IBAT_REG_ACTIVE_FLAG  IIN_REG_ACTIVE_FLAG
0x04    INT_FAULT0    0x00     II

# disconnect from the battery emulator

In [119]:
if mode_n == 3:
    iout_max = 10
elif mode_n == 2:
    iout_max = 8
elif mode_n == 1:
    iout_max = 4

list_iout = list(i for i in range(1, iout_max+1))
print("list iout :",list_iout)
ld.cc.iset = list_iout[0]
ld.enable

limit_vout = 5
limit_vin  = 20

step_coarse = 0.04
step_fine   = 0.002
# rin_path = abs((vin_init - ps.ch1.voltage) / ps.ch1.current)
# rin_path = abs((vin_init - ps.ch1.voltage) / ps.ch1.current)
rin_path = 1
print(f"RIN={rin_path:.05f}ohm")
print(f"MODE : {mode_n} to 1 (MODE={ic.MODE})")

list iout : [1, 2, 3, 4, 5, 6, 7, 8]
RIN=1.00000ohm
MODE : 2 to 1 (MODE=1)


In [3]:
ic.CP_EN=0

In [38]:
ic.ADC_EN = 0

In [120]:
now = datetime.datetime.now()
timenow = now.strftime("%Y%m%d-%H%M")
print(timenow)

20260407-1842


In [121]:
ic.write_byte(0x42, 0xc0) # disable CONV_OCP
print(f"0x42 = {ic.read_byte(0x42):#04x} (bit7=CONV_OCP)")

0x42 = 0x40 (bit7=CONV_OCP)


In [122]:
print(f"RIN={rin_path:.05f}ohm")
print(f"MODE : {mode_n} to 1 (MODE={ic.MODE})")
print(f"Efficiency at {target_vout} V")

estimate_vin = target_vout*mode_n + rin_path*(list_iout[0]/mode_n)
ps.ch1.vset = estimate_vin
set_vin = estimate_vin

ic.write_byte(0x42, 0xc0) # disable CONV_OCP
print(f"0x42 = {ic.read_byte(0x42):#04x} (bit7=CONV_OCP)")
dm1.set_nplc = 1
dm2.set_nplc = 1

log.output_set_filename(f"{timenow}-{mode_n}to1_10V_4p4V_260402Cap")

for freq in range(0,9):
    ic.FSW_SET = freq

    log.output_csv(["FSW_SET", "Frequency (kHz)", "VIN (V)", "IIN (A)", "VOUT (V)", "IOUT (A)", "Efficiency (%)", "PLoss (W)", "VIN_ADC", "VOUT_ADC", "IIN_ADC", "IBAT_ADC", "VEXT_ADC", "TDIE_ADC"])
    print(f"\nFSW_SET={freq}")

    for set_iload in list_iout:

        ld.cc.iset = set_iload
        dummuy = ps.ch1.voltage
        # dummuy = ps.ch1.voltage
        sleep(0.5)

        while True:

            sts_chg = ic.CP_SWITCHING_STAT
            if sts_chg == 0:
                print(f"CP_SWITCHING_STAT={sts_chg}, start fault handler")
                decrease_iout(target_vout=target_vout, iout_max=set_iload, mode_n=mode_n)
                
                break
            
            temp_vout = dm2.voltage_10
            diff = abs(temp_vout - target_vout)

            if (temp_vout > (target_vout-0.001)) and (temp_vout < (target_vout+0.001)):
                # print(f"break {temp_vout:.03f}, {target_vout:.03f}, {diff:.03f}")
                break

            if temp_vout < target_vout:
                
                if diff > 0.015:
                    set_vin += step_coarse
                    # print(f"low, coars, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
                else:
                    set_vin += step_fine
                    # print(f"low, fine, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
            else:
                if diff > 0.015:
                    set_vin -= step_coarse
                    # print(f"high, coars, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
                else:
                    set_vin -= (step_fine+0.001)
                    # print(f"high, fine, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
            
            ps.ch1.vset = set_vin
            # print(f"final {set_vin:.03f}")
        
         # meas_vin  = ps.ch1.voltage
        meas_vin  = dm1.voltage_100
        meas_vout = dm2.voltage_10
        meas_iin  = ps.ch1.current
        meas_iout = ld.current
        meas_freq = ds.get_meas(1) / 1000

        adc_vin  = ic.VIN_ADC * ic.lsb_vin
        adc_vout = ic.VOUT_ADC * ic.lsb_vout
        adc_iin  = ic.IIN_ADC * ic.lsb_iin
        adc_iout = ic.IBAT_ADC * ic.lsb_ibat
        adc_vext1 = ic.VEXT1_ADC * ic.lsb_vext1
        adc_tdie = ic.TDIE_ADC * ic.lsb_tdie
        
        efficiency = (meas_vout*meas_iout)/(meas_vin*meas_iin)*100
        ploss = (meas_vin*meas_iin) - (meas_vout*meas_iout)

        log.output_csv([freq, meas_freq, meas_vin, meas_iin, meas_vout, meas_iout, efficiency, ploss, adc_vin, adc_vout, adc_iin, adc_iout, adc_vext1, adc_tdie])
        print(f"FREQ={meas_freq:.01f}kHz, VIN={meas_vin:.03f}V, IIN={meas_iin:.03f}A, VOUT={meas_vout:.03f}V, IOUT={meas_iout:.03f}A, EFFICIENCY={efficiency:.03f}% (TDIE={adc_tdie:.01f}C)")

        sts_chg = ic.CP_SWITCHING_STAT
        if sts_chg == 0:
            print(f"CP_SWITCHING_STAT={sts_chg}, start fault handler")
            decrease_iout(target_vout=target_vout, iout_max=set_iload, mode_n=mode_n)
            break

    decrease_iout(target_vout=target_vout, iout_max=set_iload, mode_n=mode_n)
    log.output_csv([])

    sts_chg = ic.CP_SWITCHING_STAT
    if sts_chg == 0:
        ic.status
        ld.disable
        break

print(f'Efficiency Measure DONE!')

RIN=1.00000ohm
MODE : 2 to 1 (MODE=1)
Efficiency at 4.4 V
0x42 = 0x40 (bit7=CONV_OCP)

FSW_SET=0
FREQ=234.5kHz, VIN=8.837V, IIN=0.504A, VOUT=4.401V, IOUT=1.001A, EFFICIENCY=99.032% (TDIE=26.0C)
FREQ=234.6kHz, VIN=8.836V, IIN=1.002A, VOUT=4.382V, IOUT=2.001A, EFFICIENCY=99.019% (TDIE=26.0C)
FREQ=234.8kHz, VIN=8.909V, IIN=1.501A, VOUT=4.400V, IOUT=3.001A, EFFICIENCY=98.733% (TDIE=27.0C)
FREQ=234.8kHz, VIN=8.908V, IIN=2.000A, VOUT=4.381V, IOUT=4.001A, EFFICIENCY=98.375% (TDIE=28.5C)
FREQ=235.8kHz, VIN=8.984V, IIN=2.499A, VOUT=4.401V, IOUT=5.001A, EFFICIENCY=98.017% (TDIE=30.0C)
FREQ=236.4kHz, VIN=8.984V, IIN=2.998A, VOUT=4.381V, IOUT=6.001A, EFFICIENCY=97.627% (TDIE=32.0C)
FREQ=237.7kHz, VIN=9.061V, IIN=3.497A, VOUT=4.401V, IOUT=7.001A, EFFICIENCY=97.244% (TDIE=35.0C)
FREQ=475.9kHz, VIN=9.060V, IIN=3.995A, VOUT=4.381V, IOUT=8.001A, EFFICIENCY=96.840% (TDIE=38.0C)
VIN=9.060, ILOAD=8.0
VIN=9.027, ILOAD=7.0
VIN=8.995, ILOAD=6.0
VIN=8.962, ILOAD=5.0
VIN=8.930, ILOAD=4.0
VIN=8.897, ILOAD=3.0
V

In [105]:
ic.status

Addr    Reg           Value    Bit7                  Bit6               Bit5               Bit4                Bit3                  Bit2                  Bit1                  Bit0
------  ------------  -------  --------------------  -----------------  -----------------  ------------------  --------------------  --------------------  --------------------  -------------------
0x01    INT_DEVICE0   0x00     POR_FLAG              CP_SWITCHING_FLAG  QB_ON_FLAG         VIN_PRESENT_FLAG    VOUT_INSERT_FLAG      VOUT_TH_REV_EN_FLAG   VOUT_TH_CHG_EN_FLAG   VIN_TH_CHG_EN_FLAG
0x02    INT_DEVICE1   0x00     VEXT1_REMOVE_FLAG     VEXT2_REMOVE_FLAG  VEXT1_INSERT_FLAG  VEXT2_INSERT_FLAG   VEXT1_OVP_FLAG        VEXT2_OVP_FLAG        VEXT1_DRV_ON_FLAG     VEXT2_DRV_ON_FLAG
0x03    INT_DEVICE2   0x00     ADC_DONE_FLAG         IIN_UCP_RISE_FLAG  RESERVED           TDIE_REG_EXIT_FLAG  TDIE_REG_ACTIVE_FLAG  VBAT_REG_ACTIVE_FLAG  IBAT_REG_ACTIVE_FLAG  IIN_REG_ACTIVE_FLAG
0x04    INT_FAULT0    0x00     II

In [106]:
ld.disable
ic.CP_EN=0
# Test Done!


In [17]:
ic.status_adc

ADC        Hex      Value
---------  -----  -------
IIN_ADC    0xfff  7.67812
VIN_ADC    0xfff   20.475
VEXT1_ADC  0xfff   20.475
VEXT2_ADC  0xfff   20.475
VOUT_ADC   0xfff  5.11875
VBAT_ADC   0xfff  5.11875
IBAT_ADC   0xfff  12.7969
C1P_ADC    0xfff   20.475
NTC_ADC    0xfff  59.9917
TDIE_ADC   0x3ff    511.5


In [ ]:
hex(ic.read_byte(0))

In [ ]:
ic.i2c_scan()

In [ ]:
ic.update_i2c_address(110)